# SymBot — Launch (Transformer + RAG Hybrid)
**Pipeline:** Symptoms → Custom Transformer (classification + severity) → RAG retrieval for context → Mistral generates explanation → Gradio UI

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')

✅ Using device: cpu


## 1. Re-declare Model Architecture (must match training)

In [2]:
class SymptomTokenEmbedding(nn.Module):
    def __init__(self, num_symptoms, d_model):
        super().__init__()
        self.token_embed = nn.Embedding(num_symptoms, d_model)
        self.present_embed = nn.Parameter(torch.randn(d_model) * 0.02)
        self.absent_embed  = nn.Parameter(torch.randn(d_model) * 0.02)
        self.num_symptoms = num_symptoms
        self.d_model = d_model

    def forward(self, x):
        batch_size = x.shape[0]
        idx = torch.arange(self.num_symptoms, device=x.device).unsqueeze(0).expand(batch_size, -1)
        base = self.token_embed(idx)
        presence = x.unsqueeze(-1)
        tokens = base + presence * self.present_embed + (1 - presence) * self.absent_embed
        return tokens


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, attn_weights = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn_weights


class SymBotTransformer(nn.Module):
    def __init__(self, num_symptoms, num_classes, d_model=64, num_heads=4, d_ff=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.tokenizer = SymptomTokenEmbedding(num_symptoms, d_model)
        self.encoders = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.pool_proj = nn.Linear(d_model, d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )
        self.severity_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, 1)
        )

    def forward(self, x):
        tokens = self.tokenizer(x)
        attn_maps = []
        for encoder in self.encoders:
            tokens, attn_w = encoder(tokens)
            attn_maps.append(attn_w)
        presence = x.unsqueeze(-1)
        weighted = tokens * (presence + 0.1)
        pooled = weighted.sum(dim=1) / (presence.sum(dim=1) + 1e-6)
        pooled = F.gelu(self.pool_proj(pooled))
        disease_logits = self.classifier(pooled)
        severity_pred = self.severity_head(pooled).squeeze(-1)
        return disease_logits, severity_pred, attn_maps

print('✅ Architecture re-declared')

✅ Architecture re-declared


## 2. Load Trained Model + Vectorstore + LLM

In [3]:
checkpoint = torch.load('symbot_transformer.pt', map_location=device, weights_only=False)

ALL_SYMPTOMS = checkpoint['symptom_vocab']
SYMPTOM_TO_IDX = {s: i for i, s in enumerate(ALL_SYMPTOMS)}
NUM_SYMPTOMS = len(ALL_SYMPTOMS)
DISEASE_CLASSES = checkpoint['label_encoder_classes']
NUM_CLASSES = len(DISEASE_CLASSES)

model = SymBotTransformer(
    num_symptoms=NUM_SYMPTOMS,
    num_classes=NUM_CLASSES,
    d_model=checkpoint['d_model'],
    num_heads=checkpoint['num_heads'],
    d_ff=checkpoint['d_ff'],
    num_layers=checkpoint['num_layers'],
).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f'✅ Transformer loaded — {NUM_SYMPTOMS} symptoms, {NUM_CLASSES} diseases')

✅ Transformer loaded — 132 symptoms, 41 diseases


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaLLM

print('⏳ Loading embeddings + vectorstore...')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})
vectorstore = Chroma(persist_directory='vectorstore', embedding_function=embeddings)

print('⏳ Loading Mistral via Ollama...')
llm = OllamaLLM(model='mistral:latest', temperature=0.3)

print('✅ All systems loaded')

⏳ Loading embeddings + vectorstore...


C:\Users\alpha\AppData\Local\Temp\ipykernel_37156\1239608568.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})


C:\Users\alpha\AppData\Local\Temp\ipykernel_37156\1239608568.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory='vectorstore', embedding_function=embeddings)


⏳ Loading Mistral via Ollama...
✅ All systems loaded


## 3. Symptom Extraction from Free Text (handles Hinglish loosely via fuzzy match)

In [5]:
from difflib import get_close_matches

HINGLISH_MAP = {
    'bukhaar': 'fever', 'bukhar': 'fever', 'sar dard': 'headache', 'sir dard': 'headache',
    'badan dard': 'muscle pain', 'kamzori': 'fatigue', 'thakaan': 'fatigue',
    'khansi': 'cough', 'jukam': 'continuous sneezing', 'pet dard': 'stomach pain',
    'ulti': 'vomiting', 'matli': 'nausea', 'chakkar': 'dizziness', 'jalan': 'burning micturition',
    'khujli': 'itching', 'daane': 'skin rash', 'saans': 'breathlessness',
}

def extract_symptoms(text: str, threshold=0.6):
    text_lower = text.lower()

    for hindi, eng in HINGLISH_MAP.items():
        if hindi in text_lower:
            text_lower = text_lower.replace(hindi, eng)

    found = set()
    for symptom in ALL_SYMPTOMS:
        if symptom in text_lower:
            found.add(symptom)

    words = text_lower.replace(',', ' ').split()
    for i in range(len(words)):
        for j in range(i+1, min(i+4, len(words)+1)):
            phrase = ' '.join(words[i:j])
            matches = get_close_matches(phrase, ALL_SYMPTOMS, n=1, cutoff=threshold)
            if matches:
                found.add(matches[0])

    return list(found)

print(extract_symptoms('mujhe bukhaar hai aur sar dard ho raha hai'))
print(extract_symptoms('itching, skin rash and fever'))

['headache', 'mild fever']
['itching', 'skin rash', 'mild fever']


## 4. Full Hybrid Pipeline — Transformer Classification + RAG Explanation

In [6]:
def check_symptoms(user_input: str) -> str:
    if not user_input.strip():
        return '⚠️ Please describe your symptoms.'

    matched_symptoms = extract_symptoms(user_input)

    if not matched_symptoms:
        return ("❌ Couldn't recognize specific symptoms from your description.\n"
                 "Try mentioning specific symptoms like 'fever', 'headache', 'rash', etc.")

    vec = np.zeros(NUM_SYMPTOMS, dtype=np.float32)
    for s in matched_symptoms:
        if s in SYMPTOM_TO_IDX:
            vec[SYMPTOM_TO_IDX[s]] = 1.0

    x = torch.tensor(vec, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits, severity_pred, _ = model(x)
        probs = F.softmax(logits, dim=1).squeeze(0)
        top_probs, top_idx = probs.topk(3)

    predictions = [(DISEASE_CLASSES[i], float(p)) for p, i in zip(top_probs.cpu().numpy(), top_idx.cpu().numpy())]
    severity_score = round(float(severity_pred.item()), 2)

    # 1. Retrieve verified DB entries for ALL top 3 candidates to give context
    contexts = []
    seen_diseases = set()
    for rank, (disease, prob) in enumerate(predictions):
        docs = vectorstore.similarity_search(disease, k=1)
        if docs:
            contexts.append(docs[0].page_content)
            seen_diseases.add(docs[0].metadata.get("disease", disease))
        else:
            contexts.append(f"Disease: {disease}\nNo verified database description available.")
            seen_diseases.add(disease)

    # 2. Retrieve verified DB entries matching the specific symptoms directly
    symptom_query = ', '.join(matched_symptoms)
    symptom_docs = vectorstore.similarity_search(symptom_query, k=2)
    symptom_contexts = []
    for doc in symptom_docs:
        d_name = doc.metadata.get("disease", "")
        if d_name not in seen_diseases:
            symptom_contexts.append(doc.page_content)
            seen_diseases.add(d_name)
    
    combined_contexts = "\n\n---\n\n".join(contexts + symptom_contexts)
    pred_summary = '\n'.join([f'  {i+1}. {d} ({p*100:.1f}% confidence)' for i, (d, p) in enumerate(predictions)])

    # 3. Refined Clinical Validator Prompt
    prompt = f"""You are SymBot, a knowledgeable AI medical symptom assistant for Indian patients.

A custom Transformer model analyzed the patient's symptoms and predicted these top 3 candidate conditions:
{pred_summary}

Predicted severity score: {severity_score} / 7

Verified local database entries for candidate conditions and symptom-matched conditions:
{combined_contexts}

Patient self-description: \"{user_input}\"
Detected symptoms: {', '.join(matched_symptoms)}

YOUR ROLE:
1. Act as a Clinical Validator. Review the patient's description and detected symptoms against the verified local database entries and your own offline medical knowledge.
2. Determine which condition is the most clinically accurate match.
   - If the Transformer's top prediction is incorrect or doesn't fit the symptoms, explain why and highlight the best match from the database entries (either from the other Transformer candidates or the direct symptom-matched entries).
   - Explain the reasoning clearly based on the matching symptoms.
3. Keep the tone warm, empathetic, simple, and professional.
4. Provide a disclaimer stating you are an AI, not a replacement for a doctor.
5. Provide clear precautions based on the verified local database entry of the most likely disease.
6. If the severity score is above 4.0, strongly advise seeing a medical professional soon.

Response:"""

    response = llm.invoke(prompt)

    final_output = f"""🔍 **Detected Symptoms:** {', '.join(matched_symptoms)}

🧠 **Transformer Predictions:**
{pred_summary}

⚖️ **Severity Score:** {severity_score} / 7

---

{response}"""

    return final_output

print('✅ check_symptoms() ready')

✅ check_symptoms() ready


In [7]:
try:
    result = check_symptoms('I have fever, headache and rash on my body')
    print(result)
except Exception as e:
    import traceback
    traceback.print_exc()

🔍 **Detected Symptoms:** headache, skin rash, high fever, mild fever

🧠 **Transformer Predictions:**
  1. Bronchial Asthma (22.0% confidence)
  2. Paralysis (brain hemorrhage) (13.3% confidence)
  3. GERD (9.5% confidence)

⚖️ **Severity Score:** 4.82 / 7

---

 Based on your symptoms of fever, headache, and skin rash, I would suggest that Malaria could be a possible condition to consider. The symptoms you've mentioned are closely aligned with those associated with Malaria, such as high fever, headache, and chills. Although the Transformer model predicted Bronchial Asthma as the top candidate, it seems less likely given your symptoms do not include breathlessness, cough, or family history of asthma.

Paralysis (brain hemorrhage) and GERD are also possibilities, but their symptoms do not align as closely with your self-description. For instance, while Paralysis may cause headaches, it typically presents with weakness on one side of the body, which you have not mentioned experiencing. GE

## 5. Gradio UI

In [8]:
import gradio as gr
import socket

def find_free_port(start_port=7861):
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('127.0.0.1', port))
                return port
            except OSError:
                port += 1

EXAMPLES = [
    ['mujhe bukhaar hai, sar dard aur badan mein dard ho raha hai'],
    ['itching on skin with rash and fever'],
    ['chest pain, shortness of breath and sweating'],
    ['frequent urination, excessive thirst and fatigue'],
    ['joint pain, swelling and morning stiffness'],
    ['nausea, vomiting, yellow eyes and dark urine'],
]

with gr.Blocks(
    title='SymBot — AI Symptom Checker'
) as demo:

    gr.HTML("""
        <div class='header'>
            <h1>🩺 SymBot — AI Symptom Checker</h1>
            <p>Step 2 of MediVerse AI &nbsp;|&nbsp; Custom Transformer + Mistral RAG &nbsp;|&nbsp; 100% Offline</p>
        </div>
    """)

    gr.HTML("""
        <div class='disclaimer'>
            ⚠️ <b>Disclaimer:</b> SymBot is an AI assistant for informational purposes only.
            Always consult a qualified doctor for medical advice.
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            symptom_input = gr.Textbox(
                label='🤒 Describe Your Symptoms',
                placeholder='e.g. "fever, headache and body ache" or Hinglish: "mujhe bukhaar hai aur sar dard"',
                lines=4
            )
            with gr.Row():
                check_btn = gr.Button('🔍 Check Symptoms', variant='primary', scale=3)
                clear_btn = gr.Button('🗑️ Clear', scale=1)

            gr.Examples(examples=EXAMPLES, inputs=symptom_input, label='💡 Try These Examples')

        with gr.Column(scale=1):
            result_output = gr.Markdown(label='🩺 SymBot Analysis')

    with gr.Accordion('⚙️ Tech Stack', open=False):
        gr.Markdown("""
        | Component | Technology |
        |-----------|------------|
        | 🧠 Classifier | Custom Transformer (PyTorch, self-attention, GELU) |
        | 📊 Severity Predictor | Regression head, manual backprop training |
        | 🗨️ Explanation LLM | Mistral 7B (Ollama) |
        | 🔍 Embeddings | MiniLM-L6-v2 |
        | 📚 Vector DB | ChromaDB |
        | 🔗 Orchestration | LangChain |
        | 🌐 Internet | Not required |
        """)

    gr.HTML("<div class='footer'>SymBot — Step 2 of MediVerse AI &nbsp;|&nbsp; Built by Jashan &nbsp;|&nbsp; JECRC University</div>")

    check_btn.click(fn=check_symptoms, inputs=symptom_input, outputs=result_output)
    clear_btn.click(fn=lambda: ('', ''), outputs=[symptom_input, result_output])

free_port = find_free_port(7861)
print(f"🚀 Launching Gradio on port {free_port}...")
demo.launch(
    share=False,
    server_port=free_port,
    theme=gr.themes.Soft(primary_hue='emerald'),
    css="""
        .header { text-align: center; padding: 20px 0; }
        .header h1 { font-size: 2em; color: #059669; }
        .disclaimer { background: #fef3c7; border-left: 4px solid #f59e0b;
                      padding: 10px 15px; border-radius: 6px; font-size: 0.85em; }
        .footer { text-align: center; color: #6b7280; font-size: 0.8em; margin-top: 20px; }
    """
)

🚀 Launching Gradio on port 7862...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
